# TP Jour 3 — Spark SQL Avancé, Rigueur Statistique & Couche Gold

### Objectifs de la séance
1. Diagnostiquer l'**asymétrie (`SKEWNESS`)** de vos variables quantitatives et justifier l'emploi d'indicateurs robustes (**Médiane `PERCENTILE_APPROX`**, **IQR `P75 - P25`**, **P90**) plutôt que la seule moyenne arithmétique.
2. Écrire des requêtes **Spark SQL avancées** mobilisant des CTE (`WITH ...`) et des **fonctions de fenêtrage** (`DENSE_RANK`, `ROW_NUMBER`, cumul de **Pareto**, variation temporelle **`LAG`**).
3. Construire au moins **2 à 3 tables agrégées métier dans `/lakehouse/gold/`** prêtes à alimenter votre Dashboard Streamlit à faible latence.
4. Éviter le *Small Files Problem* sur HDFS et compacter votre table Silver avec `OPTIMIZE`.

In [1]:
from pyspark.sql import SparkSession
from delta.tables import DeltaTable

spark = (
    SparkSession.builder
    .appName("J3_Analytics_SQL_Gold")
    .getOrCreate()
)

SILVER_MAIN_PATH = "/lakehouse/silver/flights_enriched"
spark.read.format("delta").load(SILVER_MAIN_PATH).createOrReplaceTempView("v_silver")
print("Lignes disponibles dans v_silver :", spark.sql("SELECT COUNT(*) FROM v_silver").first()[0])
spark.sql("SELECT * FROM v_silver WHERE cancelled = 0 AND arrival_delay IS NOT NULL").createOrReplaceTempView("v_observed")
print("Les analyses de retard portent sur les vols non annulés avec retard à l'arrivée connu.")

:: loading settings :: url = jar:file:/opt/venv/lib/python3.15/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-8e762689-58ef-426c-aae5-8f1556f8f99e;1.0
	confs: [default]


	found io.delta#delta-spark_4.2_2.13;4.4.1 in central
	found io.delta#delta-storage;4.4.1 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central


	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	found io.delta#delta-kernel-api;4.4.1 in central
	found org.roaringbitmap#RoaringBitmap;0.9.25 in central
	found com.fasterxml.jackson.core#jackson-databind;2.13.5 in central
	found com.fasterxml.jackson.core#jackson-annotations;2.13.5 in central
	found com.fasterxml.jackson.core#jackson-core;2.13.5 in central
	found com.fasterxml.jackson.datatype#jackson-datatype-jdk8;2.13.5 in central
	found org.roaringbitmap#shims;0.9.25 in central


	found io.delta#delta-kernel-defaults;4.4.1 in central
	found org.apache.hadoop#hadoop-client-runtime;3.4.2 in central
	found org.apache.parquet#parquet-hadoop;1.16.0 in central
	found org.apache.parquet#parquet-column;1.16.0 in central
	found org.apache.parquet#parquet-common;1.16.0 in central
	found org.apache.parquet#parquet-format-structures;1.16.0 in central
	found javax.annotation#javax.annotation-api;1.3.2 in central
	found org.apache.parquet#parquet-encoding;1.16.0 in central


	found org.locationtech.jts#jts-core;1.20.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.7 in central
	found io.airlift#aircompressor;2.0.2 in central
	found commons-pool#commons-pool;1.6 in central
	found com.github.luben#zstd-jni;1.5.7-3 in central
	found org.apache.hadoop#hadoop-client-api;3.4.2 in central
	found commons-logging#commons-logging;1.3.0 in central
	found org.apache.parquet#parquet-jackson;1.16.0 in central
	found io.delta#delta-kernel-unitycatalog;4.4.1 in central
:: resolution report :: resolve 950ms :: artifacts dl 29ms
	:: modules in use:
	com.fasterxml.jackson.core#jackson-annotations;2.13.5 from central in [default]
	com.fasterxml.jackson.core#jackson-core;2.13.5 from central in [default]
	com.fasterxml.jackson.core#jackson-databind;2.13.5 from central in [default]


	com.fasterxml.jackson.datatype#jackson-datatype-jdk8;2.13.5 from central in [default]
	com.github.luben#zstd-jni;1.5.7-3 from central in [default]
	com.google.code.findbugs#jsr305;3.0.2 from central in [default]
	commons-logging#commons-logging;1.3.0 from central in [default]
	commons-pool#commons-pool;1.6 from central in [default]
	io.airlift#aircompressor;2.0.2 from central in [default]
	io.delta#delta-kernel-api;4.4.1 from central in [default]
	io.delta#delta-kernel-defaults;4.4.1 from central in [default]
	io.delta#delta-kernel-unitycatalog;4.4.1 from central in [default]
	io.delta#delta-spark_4.2_2.13;4.4.1 from central in [default]
	io.delta#delta-storage;4.4.1 from central in [default]
	io.unitycatalog#unitycatalog-client;0.6.0 from central in [default]
	io.unitycatalog#unitycatalog-hadoop;0.6.0 from central in [default]
	javax.annotation#javax.annotation-api;1.3.2 from central in [default]
	org.apache.hadoop#hadoop-client-api;3.4.2 from central in [default]
	org.apache.hadoop#

... (2 outputs left out)

/opt/venv/lib/python3.15/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


26/11/17 22:03:08 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


26/11/17 22:03:17 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Lignes disponibles dans v_silver : 5819079
Les analyses de retard portent sur les vols non annulés avec retard à l'arrivée connu.


## 1. Analyse d'Asymétrie : Moyenne vs Médiane & Écart Interquartile (IQR)

### Exercice 1.1 — Étudier la distribution de votre métrique quantitative principale
Écrivez une requête Spark SQL sur `v_silver` calculant sur votre variable continue principale :
- `COUNT(*)`, `AVG(...)`, `STDDEV(...)`, `SKEWNESS(...)`
- Les quantiles `P25`, `P50` (médiane), `P75`, `P90`, `P95` via `PERCENTILE_APPROX(col, q)` et `MAX(...)`.

Rédigez ensuite en Markdown **votre interprétation statistique** (écart moyenne/médiane, impact des valeurs extrêmes, choix de l'indicateur retenu pour votre dashboard).

In [2]:
df_dist_summary = spark.sql("""
SELECT COUNT(*) AS n_vols, AVG(arrival_delay) AS moyenne,
       STDDEV(arrival_delay) AS ecart_type, SKEWNESS(arrival_delay) AS asymetrie,
       PERCENTILE_APPROX(arrival_delay, 0.25, 10000) AS p25,
       PERCENTILE_APPROX(arrival_delay, 0.50, 10000) AS mediane,
       PERCENTILE_APPROX(arrival_delay, 0.75, 10000) AS p75,
       PERCENTILE_APPROX(arrival_delay, 0.90, 10000) AS p90,
       PERCENTILE_APPROX(arrival_delay, 0.95, 10000) AS p95,
       MAX(arrival_delay) AS maximum
FROM v_observed
""")
df_dist_summary.show(truncate=False)
distribution = df_dist_summary.first()
if distribution.n_vols < 2 or distribution.asymetrie is None:
    raise ValueError("Distribution insuffisante pour interpréter l'asymétrie.")
print(f"Moyenne : {distribution.moyenne:.2f} min ; médiane : {distribution.mediane} min ; IQR : {distribution.p75 - distribution.p25} min ; asymétrie : {distribution.asymetrie:.2f}.")

+-------+-----------------+-----------------+-----------------+---+-------+---+---+---+-------+
|n_vols |moyenne          |ecart_type       |asymetrie        |p25|mediane|p75|p90|p95|maximum|
+-------+-----------------+-----------------+-----------------+---+-------+---+---+---+-------+
|5714008|4.407057357987598|39.27129709388633|6.502894504785056|-13|-5     |8  |34 |66 |1971   |
+-------+-----------------+-----------------+-----------------+---+-------+---+---+---+-------+



Moyenne : 4.41 min ; médiane : -5 min ; IQR : 21 min ; asymétrie : 6.50.


**Justification statistique :** une asymétrie positive et une moyenne supérieure à la médiane indiquent que les retards élevés tirent la moyenne vers le haut. La médiane et l'IQR décrivent le vol typique et sa dispersion sans être dominés par les extrêmes ; P90/P95 rendent visibles les retards importants. Les résultats exacts sont affichés ci-dessus.

---
## 2. Table Gold #1 : Classement Métier & Loi de Pareto (`OVER ORDER BY ...`)

### Exercice 2.1 — Agrégation par segment/catégorie + Courbe cumulée de Pareto
Construisez en Spark SQL une première table Gold agrégée par une dimension métier pertinente de votre dataset incluant :
- Le volume d'observations, le total et les indicateurs robustes (**médiane**, **IQR**).
- Le rang via `DENSE_RANK() OVER (ORDER BY ... DESC)`.
- Le pourcentage cumulé du total (**analyse de Pareto 80/20**) via :
  `SUM(metric_total) OVER (ORDER BY metric_total DESC ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) / SUM(metric_total) OVER ()`.

Ici, le Pareto porte sur la somme des **minutes de retard positives** : une arrivée en avance contribue zéro. Ce choix évite que les avances annulent les retards et conserve un cumul croissant. Les vols annulés et les retards inconnus sont exclus de ce calcul.

In [3]:
GOLD_PARETO_PATH = "/lakehouse/gold/flights_airline_pareto"
df_gold_1 = spark.sql("""
WITH grouped AS (
  SELECT airline, MAX(airline_name) AS airline_name, COUNT(*) AS n_vols,
         SUM(GREATEST(arrival_delay, 0)) AS minutes_retard,
         AVG(arrival_delay) AS moyenne, PERCENTILE_APPROX(arrival_delay, 0.25, 10000) AS p25,
         PERCENTILE_APPROX(arrival_delay, 0.50, 10000) AS mediane,
         PERCENTILE_APPROX(arrival_delay, 0.75, 10000) AS p75,
         PERCENTILE_APPROX(arrival_delay, 0.90, 10000) AS p90,
         CAST(100 AS DOUBLE) * AVG(CASE WHEN arrival_delay >= 15 THEN CAST(1 AS DOUBLE) ELSE CAST(0 AS DOUBLE) END) AS retard_pct
  FROM v_observed GROUP BY airline
), ranked AS (
  SELECT *, p75 - p25 AS iqr,
         DENSE_RANK() OVER (ORDER BY minutes_retard DESC) AS rang,
         SUM(minutes_retard) OVER (ORDER BY minutes_retard DESC, airline ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cumul,
         SUM(minutes_retard) OVER () AS total
  FROM grouped
)
SELECT airline, airline_name, n_vols, minutes_retard, moyenne, p25, mediane, p75, p90, iqr, retard_pct, rang,
       CASE WHEN total > 0 THEN CAST(100 AS DOUBLE) * cumul / total END AS pareto_pct
FROM ranked
""")
df_gold_1.coalesce(1).write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(GOLD_PARETO_PATH)
df_gold_1.orderBy("rang", "airline").show(15, truncate=False)

26/11/17 22:04:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 22:04:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 22:04:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 22:04:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 22:04:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 22:04:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 2

26/11/17 22:04:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 22:04:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 22:04:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 22:04:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 22:04:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 22:04:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/11/17 22:04:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 22:04:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 22:04:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 22:04:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 22:04:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 22:04:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/11/17 22:04:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 22:04:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 22:04:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 22:04:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 22:04:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 22:04:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 2

26/11/17 22:04:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 22:04:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 22:04:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 22:04:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 22:04:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 22:04:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-------+----------------------------+-------+--------------+-------------------+---+-------+---+---+---+------------------+----+------------------+
|airline|airline_name                |n_vols |minutes_retard|moyenne            |p25|mediane|p75|p90|iqr|retard_pct        |rang|pareto_pct        |
+-------+----------------------------+-------+--------------+-------------------+---+-------+---+---+---+------------------+----+------------------+
|WN     |Southwest Airlines Co.      |1242403|13849257      |4.3749636792570525 |-12|-4     |8  |33 |20 |19.04583295436344 |1   |20.041363541663607|
|AA     |American Airlines Inc.      |712935 |8611910       |3.4513721447256764 |-15|-6     |7  |33 |22 |18.273615406734134|2   |32.503723009618334|
|DL     |Delta Air Lines Inc.        |870275 |8046301       |0.18675361236390797|-15|-8     |3  |22 |18 |13.561575364109046|3   |44.14758545701543 |
|EV     |Atlantic Southeast Airlines |554752 |7504821       |6.585378691739733  |-12|-4     |9  |39 |21 |1

26/11/17 22:04:50 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 22:04:50 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 22:04:50 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 22:04:50 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 22:04:50 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 22:04:50 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


## 3. Table Gold #2 : Dynamique Temporelle & Croissance avec `LAG()`

### Exercice 3.1 — Série temporelle et taux d'évolution période-sur-période
Agrégez votre table `v_silver` selon son axe temporel (jour, semaine, mois ou année selon votre sujet) et utilisez la fonction de fenêtrage **`LAG(metric, 1) OVER (ORDER BY periode)`** (ou une moyenne glissante `AVG(...) OVER (ORDER BY periode ROWS BETWEEN ... PRECEDING AND CURRENT ROW)`) pour mesurer la dynamique d'évolution.

Sauvegardez le résultat dans `/lakehouse/gold/<nom_gold_2>`.

In [4]:
GOLD_TIME_PATH = "/lakehouse/gold/flights_monthly"
df_gold_2 = spark.sql("""
WITH monthly AS (
  SELECT DATE_TRUNC('MONTH', flight_date) AS mois, COUNT(*) AS n_vols,
         AVG(arrival_delay) AS moyenne,
         PERCENTILE_APPROX(arrival_delay, 0.50, 10000) AS mediane,
         CAST(100 AS DOUBLE) * AVG(CASE WHEN arrival_delay >= 15 THEN CAST(1 AS DOUBLE) ELSE CAST(0 AS DOUBLE) END) AS retard_pct
  FROM v_observed GROUP BY DATE_TRUNC('MONTH', flight_date)
), previous AS (
  SELECT *, LAG(n_vols) OVER (ORDER BY mois) AS vols_mois_precedent FROM monthly
)
SELECT *, CASE WHEN vols_mois_precedent > 0 THEN CAST(100 AS DOUBLE) * (n_vols - vols_mois_precedent) / vols_mois_precedent END AS evolution_vols_pct
FROM previous
""")
df_gold_2.coalesce(1).write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(GOLD_TIME_PATH)
df_gold_2.orderBy("mois").show(20, truncate=False)

26/11/17 22:04:51 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 22:04:51 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 22:04:51 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/11/17 22:04:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 22:04:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/11/17 22:04:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 22:04:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/11/17 22:05:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 22:05:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 22:05:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/11/17 22:05:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 22:05:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-------------------+------+-------------------+-------+------------------+-------------------+-------------------+
|mois               |n_vols|moyenne            |mediane|retard_pct        |vols_mois_precedent|evolution_vols_pct |
+-------------------+------+-------------------+-------+------------------+-------------------+-------------------+
|2015-01-01 00:00:00|457013|5.813582983416227  |-4     |20.995245211842988|NULL               |NULL               |
|2015-02-01 00:00:00|407663|8.320500020850556  |-3     |23.347470827619873|457013             |-10.798379914794547|
|2015-03-01 00:00:00|492138|4.92067265685641   |-4     |19.395372842576673|407663             |20.721772640636996 |
|2015-04-01 00:00:00|479251|3.1631900611579318 |-5     |17.16157086787508 |492138             |-2.6185744648858655|
|2015-05-01 00:00:00|489641|4.485018615679651  |-5     |18.308311599723062|479251             |2.1679662640244883 |
|2015-06-01 00:00:00|492847|9.601590351569554  |-3     |23.4843673594442

26/11/17 22:05:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 22:05:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


## 4. Table Gold #3 (Axe complémentaire libre) & Compaction `OPTIMIZE`

### Exercice 4.1 — Produire un 3e axe d'analyse (`ROW_NUMBER() OVER (PARTITION BY ...)`) et compacter Silver
1. Créez une 3e table Gold répondant à une question spécifique de votre problématique (ex. top sous-catégorie par région/groupe via `ROW_NUMBER() OVER (PARTITION BY groupe ORDER BY ... DESC) = 1`).
2. Exécutez `OPTIMIZE delta.'{SILVER_MAIN_PATH}'`. Les nombreux petits fichiers augmentent les métadonnées gérées par le NameNode et le nombre de tâches ; la compaction regroupe les fichiers sans changer les lignes. Les anciennes versions restent disponibles tant qu'elles ne sont pas supprimées par une opération de maintenance.

In [5]:
GOLD_AIRPORT_PATH = "/lakehouse/gold/flights_airline_top_origin"
df_gold_3 = spark.sql("""
WITH grouped AS (
  SELECT airline, origin_airport, MAX(origin_airport_name) AS airport_name,
         COUNT(*) AS n_vols, PERCENTILE_APPROX(arrival_delay, 0.50, 10000) AS mediane
  FROM v_observed GROUP BY airline, origin_airport
), ranked AS (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY airline ORDER BY n_vols DESC, origin_airport) AS position
  FROM grouped
)
SELECT * FROM ranked WHERE position = 1
""")
df_gold_3.coalesce(1).write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(GOLD_AIRPORT_PATH)
df_gold_3.orderBy("airline").show(20, truncate=False)

silver_delta = DeltaTable.forPath(spark, SILVER_MAIN_PATH)
before_files = silver_delta.detail().select("numFiles").first()[0]
spark.conf.set("spark.databricks.delta.optimize.maxFileSize", str(128 * 1024 * 1024))
spark.sql(f"OPTIMIZE delta.`{SILVER_MAIN_PATH}`").show(truncate=False)
after_files = silver_delta.detail().select("numFiles").first()[0]
print(f"Fichiers Silver actifs : {before_files} avant, {after_files} après OPTIMIZE.")

+-------+--------------+----------------------------------------------------------------------+------+-------+--------+
|airline|origin_airport|airport_name                                                          |n_vols|mediane|position|
+-------+--------------+----------------------------------------------------------------------+------+-------+--------+
|AA     |DFW           |Dallas/Fort Worth International Airport                               |131159|-5     |1       |
|AS     |SEA           |Seattle-Tacoma International Airport                                  |49806 |-3     |1       |
|B6     |JFK           |John F. Kennedy International Airport (New York International Airport)|39587 |-5     |1       |
|DL     |ATL           |Hartsfield-Jackson Atlanta International Airport                      |220400|-7     |1       |
|EV     |IAH           |George Bush Intercontinental Airport                                  |57074 |-3     |1       |
|F9     |DEN           |Denver Internati

+------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|path                                                  |metrics                                                                                                                                                                                               |
+------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|hdfs://namenode:9000/lakehouse/silver/flights_enriched|{5, 50, {123761833, 124800248, 1.242606762E8, 5, 621303381}, {12596622, 12741354, 1.266970842E7, 50, 633485421}, 1, NULL, NULL, 5, 1, 50, 0, false, 0, 0, 1794953188231, 0, 10, 

Fichiers Silver actifs : 50 avant, 5 après OPTIMIZE.
